# 04b - Assign player sex

**Input:** `data/raw/match.csv`
**What it does:** derives each player's sex (M/W) from match co-occurrence. Matches in this dataset are always men's or women's singles (never mixed), so two players in the same match share a sex. A few known-sex anchors seed a breadth-first propagation through the co-occurrence graph.
**Output:** `data/processed/player_sex.csv` (one row per player: name, sex).

This is needed because clustering all 35 players together separates men from women (a physical confound) rather than playstyle, so notebook 05 clusters within each sex.

In [1]:
## --- path bootstrap: run from the repo root no matter where this is launched ---
## nbconvert and some editors set the working directory to the notebook's own
## folder. Walk up until we find the repo root (the folder containing data/),
## chdir there so relative data paths resolve, and put code/ on sys.path so the
## shared modules (utils.py, shot_translations.py) import cleanly.
import os, sys
_d = os.getcwd()
for _ in range(5):
    if os.path.isdir(os.path.join(_d, "data")):
        break
    _d = os.path.dirname(_d)
os.chdir(_d)
if os.path.join(_d, "code") not in sys.path:
    sys.path.insert(0, os.path.join(_d, "code"))
print("working directory:", os.getcwd())

working directory: /Users/aakankshvaidya/Desktop/qss20_final_project


In [2]:
import os
import pandas as pd

In [3]:
## CONFIG
RAW_DIR = "data/raw"
PROC_DIR = "data/processed"
MATCH_PATH = os.path.join(RAW_DIR, "match.csv")
OUT_PATH = os.path.join(PROC_DIR, "player_sex.csv")
os.makedirs(PROC_DIR, exist_ok=True)

## unambiguous anchors used to seed the propagation
SEX_SEEDS = {
    "Anders ANTONSEN": "M",
    "Viktor AXELSEN":  "M",
    "Carolina MARIN":  "W",
    "An Se Young":     "W",
}

## Build the co-occurrence adjacency graph\nEach match adds an edge between its two players.

In [4]:
matches = pd.read_csv(MATCH_PATH)
print("matches:", len(matches))

adjacency = {}
for _, row in matches.iterrows():
    w, l = row["winner"], row["loser"]
    adjacency.setdefault(w, set()).add(l)
    adjacency.setdefault(l, set()).add(w)
print("unique players in adjacency:", len(adjacency))

matches: 58
unique players in adjacency: 35


## Propagate sex labels by BFS from the seeds

In [5]:
sex_assignment = {}

def propagate_from_seed(seed_name, seed_sex):
    ## breadth-first spread of one seed's sex through shared matches
    if seed_name not in adjacency:
        print(f"WARNING: seed {seed_name} not found; skipping")
        return
    queue = [seed_name]
    while queue:
        current = queue.pop(0)
        if current in sex_assignment:
            if sex_assignment[current] != seed_sex:
                print(f"CONFLICT: {current} labeled {sex_assignment[current]} but seed implies {seed_sex}")
            continue
        sex_assignment[current] = seed_sex
        for neighbor in adjacency[current]:
            if neighbor not in sex_assignment:
                queue.append(neighbor)

for seed_name, seed_sex in SEX_SEEDS.items():
    propagate_from_seed(seed_name, seed_sex)

print("players labeled:", len(sex_assignment))
print("players NOT labeled:", len(adjacency) - len(sex_assignment))
unlabeled = [p for p in adjacency if p not in sex_assignment]
if unlabeled:
    print("unlabeled (need another seed):", unlabeled)

players labeled: 35
players NOT labeled: 0


## Save

In [6]:
result = pd.DataFrame(
    [{"player_name": name, "sex": sex} for name, sex in sorted(sex_assignment.items())]
)
result.to_csv(OUT_PATH, index=False)
print(result["sex"].value_counts())
print("saved:", OUT_PATH)
print(result.head(10).to_string(index=False))

sex
M    22
W    13
Name: count, dtype: int64
saved: data/processed/player_sex.csv
             player_name sex
        Aakarshi KASHYAP   W
         Akane YAMAGUCHI   W
             An Se Young   W
         Anders ANTONSEN   M
Anthony Sinisuka GINTING   M
  Busanan ONGBAMRUNGPHAN   W
              CHEN Yufei   W
          Carolina MARIN   W
  Chico Aura DWI WARDOYO   M
Gregoria Mariska TUNJUNG   W
